In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
bidirectional_from_folders.py
-----------------------------
フォルダに既に存在する CSV 群から、4.2.4.4 の「順方向（OH→FP）×逆方向（FP→OH）」の
結合・相関・図をまとめて生成するユーティリティ。

【探すもの】
- 逆方向（FP→OH）: reverse/analysis_csv/ 内の per-condition CSV 群
  - Table_4.2.4.4R_fragment-to-OHmajor_{mode}_{index}.csv
  - Table_4.2.4.4R_fragment-pair_cohesion_{mode}_{index}.csv
  - Table_4.2.4.4R_FPcluster_cohesion_{mode}_{index}.csv
- 順方向（OH→FP）: 本文表 CSV
  - Table_4.2.4.4_OHFP-contrast_summary_k-le30*.csv

【出力（デフォルト）】
  ./paper_4.2.4.4_OHFP/bidirectional_summary/
    ├─ Table_4.2.4.4_bidirectional_joined.csv
    ├─ Table_4.2.4.4_bidirectional_correlations.csv
    ├─ Fig_4.2.4.4_bidirectional_* .png / .pdf
    └─ README_bidirectional_summary.json

【使い方例】
  # 1) 何も指定しない（カレント配下を再帰探索）
  python bidirectional_from_folders.py

  # 2) 探索を速くするためにベースを指定
  python bidirectional_from_folders.py --base /path/to/project_root

  # 3) 明示的にフォルダ/ファイルを指定
  python bidirectional_from_folders.py \
      --reverse-dir /path/to/paper_4.2.4.4_OHFP/reverse/analysis_csv \
      --forward-file /path/to/Table_4.2.4.4_OHFP-contrast_summary_k-le30.csv \
      --out /path/to/outdir --save-zip
"""

from __future__ import annotations
import argparse, sys, os, re, json, shutil
from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")  # 非GUI環境でOK
import matplotlib.pyplot as plt

# ------------------------------
# 探索ユーティリティ
# ------------------------------
FWD_PAT = re.compile(r"^Table_4\.2\.4\.4_OHFP-contrast_summary_k-le30.*\.csv$")
REV_PAT = re.compile(
    r"^Table_4\.2\.4\.4R_(fragment-to-OHmajor|fragment-pair_cohesion|FPcluster_cohesion)_(top3|cumeig)_(silhouette|dunn|gap|ch|db|ptbiserial)\.csv$"
)

def find_forward_csv(base: Path) -> Path | None:
    cands = []
    for p in base.rglob("*.csv"):
        if FWD_PAT.match(p.name):
            cands.append(p)
    if not cands:
        return None
    return sorted(cands, key=os.path.getmtime)[-1]

def find_reverse_analysis_dir(base: Path) -> Path | None:
    """
    base 配下から reverse/analysis_csv を**優先的**に見つける。
    無ければ、REV_PAT にマッチする CSV が多数あるフォルダを推測する。
    """
    # 優先パターン
    for p in base.rglob("analysis_csv"):
        # 逆方向 CSV が複数ある場所を優先
        files = list(p.glob("Table_4.2.4.4R_*.csv"))
        ok = sum(1 for f in files if REV_PAT.match(f.name)) >= 3
        if ok:
            return p

    # フォールバック：マッチ CSV が多いフォルダをスコアリング
    best_dir, best_cnt = None, 0
    for p in base.rglob("*.csv"):
        if REV_PAT.match(p.name):
            cnt = len(list(p.parent.glob("Table_4.2.4.4R_*.csv")))
            if cnt > best_cnt:
                best_dir, best_cnt = p.parent, cnt
    return best_dir

# ------------------------------
# 逆方向サマリーの再構築
# ------------------------------
def rebuild_reverse_summary(analysis_csv_dir: Path) -> pd.DataFrame:
    pat = re.compile(
        r"Table_4\.2\.4\.4R_(?P<kind>fragment-to-OHmajor|fragment-pair_cohesion|FPcluster_cohesion)"
        r"_(?P<mode>top3|cumeig)_(?P<index>silhouette|dunn|gap|ch|db|ptbiserial)\.csv$"
    )
    by_key = {}
    for f in analysis_csv_dir.glob("Table_4.2.4.4R_*.csv"):
        m = pat.match(f.name)
        if not m:
            continue
        kind = m["kind"]; mode = m["mode"]; index = m["index"]
        key = (mode, index)
        by_key.setdefault(key, {"mode": mode, "index": index})
        df = pd.read_csv(f)

        if kind == "fragment-to-OHmajor":
            pur = pd.to_numeric(df.get("OH_purity"), errors="coerce")
            ent = pd.to_numeric(df.get("OH_entropy"), errors="coerce")
            by_key[key]["mean_OH_purity"]  = float(pur.dropna().mean()) if pur.notna().any() else np.nan
            by_key[key]["mean_OH_entropy"] = float(ent.dropna().mean()) if ent.notna().any() else np.nan
            by_key[key]["n_fragments"]     = int(df.shape[0])

        elif kind == "fragment-pair_cohesion":
            ms = df.get("OH_major_same")
            if ms is not None:
                ms2 = ms.copy()
                if ms2.dtype == "O":
                    ms2 = ms2.astype(str).str.lower().map({"true": True, "false": False})
                by_key[key]["pair_OH_major_same_rate"] = float(pd.to_numeric(ms2, errors="coerce").mean())
            cs = pd.to_numeric(df.get("OH_cosine_similarity"), errors="coerce")
            js = pd.to_numeric(df.get("OH_JS_divergence"),   errors="coerce")
            by_key[key]["pair_mean_cosine_similarity"] = float(cs.dropna().mean()) if cs.notna().any() else np.nan
            by_key[key]["pair_mean_JS_divergence"]     = float(js.dropna().mean()) if js.notna().any() else np.nan

        elif kind == "FPcluster_cohesion":
            cr = pd.to_numeric(df.get("cohesive_ratio"), errors="coerce")
            by_key[key]["FPcluster_median_cohesive_ratio"] = float(cr.dropna().median()) if cr.notna().any() else np.nan

    return pd.DataFrame(sorted(by_key.values(), key=lambda d:(d["mode"], d["index"])))

# ------------------------------
# 順方向の横持ち化 + 平均列
# ------------------------------
def make_forward_wide(fwd_csv: Path) -> pd.DataFrame:
    fwd = pd.read_csv(fwd_csv)
    keep = ["mode","index","Dataset","ARI","NMI","AMI","kA","kB","n"]
    for c in keep:
        if c not in fwd.columns:
            fwd[c] = np.nan
    w = (fwd[keep]
         .pivot_table(index=["mode","index"],
                      columns="Dataset",
                      values=["ARI","NMI","AMI","kA","kB","n"],
                      aggfunc="first"))
    w.columns = [f"{a}_{b}" for a,b in w.columns]
    w = w.reset_index()
    # 列名の _oh/_fp を大文字に
    w = w.rename(columns={c: c.replace("_oh","_OH").replace("_fp","_FP") for c in w.columns})

    def mean2(a, b):
        a = pd.to_numeric(a, errors="coerce")
        b = pd.to_numeric(b, errors="coerce")
        return pd.concat([a,b], axis=1).mean(axis=1, skipna=True)

    if {"ARI_OH","ARI_FP"}.issubset(w.columns): w["ARI_mean"] = mean2(w["ARI_OH"], w["ARI_FP"])
    if {"NMI_OH","NMI_FP"}.issubset(w.columns): w["NMI_mean"] = mean2(w["NMI_OH"], w["NMI_FP"])
    if {"AMI_OH","AMI_FP"}.issubset(w.columns): w["AMI_mean"] = mean2(w["AMI_OH"], w["AMI_FP"])
    return w

# ------------------------------
# 相関 & 図
# ------------------------------
def compute_correlations(df_merged: pd.DataFrame) -> pd.DataFrame:
    import scipy.stats as stats
    rev_metrics = [
        "pair_OH_major_same_rate",
        "pair_mean_cosine_similarity",
        "pair_mean_JS_divergence",
        "FPcluster_median_cohesive_ratio",
        "mean_OH_purity", "mean_OH_entropy"
    ]
    f_metrics = [c for c in ["ARI_mean","NMI_mean","AMI_mean"] if c in df_merged.columns]
    rows=[]
    for fm in f_metrics:
        for rm in rev_metrics:
            if rm not in df_merged.columns:
                continue
            x = pd.to_numeric(df_merged[fm], errors="coerce")
            y = pd.to_numeric(df_merged[rm], errors="coerce")
            m = x.notna() & y.notna()
            if m.sum() >= 3:
                pr = stats.pearsonr(x[m], y[m])
                sr = stats.spearmanr(x[m], y[m])
                rows.append({
                    "forward_metric": fm, "reverse_metric": rm, "n": int(m.sum()),
                    "pearson_r": float(pr.statistic), "pearson_p": float(pr.pvalue),
                    "spearman_rho": float(sr.statistic), "spearman_p": float(sr.pvalue)
                })
            else:
                rows.append({
                    "forward_metric": fm, "reverse_metric": rm, "n": int(m.sum()),
                    "pearson_r": np.nan, "pearson_p": np.nan,
                    "spearman_rho": np.nan, "spearman_p": np.nan
                })
    return pd.DataFrame(rows)

def scatter(ax, x, y, title=""):
    ax.scatter(x, y, alpha=0.9)
    if len(x) >= 2 and len(y) >= 2:
        coef = np.polyfit(x, y, 1)
        xs = np.linspace(float(np.min(x)), float(np.max(x)), 100)
        ys = np.polyval(coef, xs)
        ax.plot(xs, ys)
    ax.set_title(title)
    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)

def save_core_scatter(df, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    pairs = [
        ("ARI_mean","pair_OH_major_same_rate","ARI_mean vs pair_OH_major_same_rate"),
        ("ARI_mean","pair_mean_cosine_similarity","ARI_mean vs pair_mean_cosine_similarity"),
        ("ARI_mean","FPcluster_median_cohesive_ratio","ARI_mean vs FPcluster_median_cohesive_ratio"),
    ]
    for xcol, ycol, ttl in pairs:
        if xcol not in df.columns or ycol not in df.columns:
            continue
        x = pd.to_numeric(df[xcol], errors="coerce")
        y = pd.to_numeric(df[ycol], errors="coerce")
        m = x.notna() & y.notna()
        if m.sum() < 2:
            continue
        fig, ax = plt.subplots(figsize=(6,4), dpi=300)
        scatter(ax, x[m].values, y[m].values, ttl)
        ax.set_xlabel(xcol); ax.set_ylabel(ycol)
        fig.tight_layout()
        fig.savefig(out_dir / f"Fig_4.2.4.4_bidirectional_{xcol}_vs_{ycol}.png", bbox_inches="tight")
        fig.savefig(out_dir / f"Fig_4.2.4.4_bidirectional_{xcol}_vs_{ycol}.pdf", bbox_inches="tight")
        plt.close(fig)

# ------------------------------
# MAIN
# ------------------------------
def main():
    ap = argparse.ArgumentParser(description="4.2.4.4 bidirectional builder from folders")
    ap.add_argument("--base", type=str, default=".", help="探索の起点フォルダ（省略時: カレント）")
    ap.add_argument("--reverse-dir", type=str, default=None, help="逆方向 per-condition CSV が入った analysis_csv フォルダを直接指定")
    ap.add_argument("--forward-file", type=str, default=None, help="順方向本文表 Table_4.2.4.4_OHFP-contrast_summary_k-le30*.csv を直接指定")
    ap.add_argument("--forward-dir", type=str, default=None, help="順方向 CSV の探索フォルダ（指定が無ければ --base を再帰探索）")
    ap.add_argument("--out", type=str, default=None, help="出力フォルダ（省略時: ./paper_4.2.4.4_OHFP/bidirectional_summary）")
    ap.add_argument("--save-zip", action="store_true", help="出力一式を zip 化")
    args, _ = ap.parse_known_args()  # Jupyter 未知引数対策

    base = Path(args.base).resolve()
    out_root = Path(args.out).resolve() if args.out else (base / "paper_4.2.4.4_OHFP" / "bidirectional_summary")
    out_root.mkdir(parents=True, exist_ok=True)

    # 1) 逆方向 analysis_csv を決定
    if args.reverse_dir:
        rev_dir = Path(args.reverse_dir).resolve()
        if not rev_dir.exists():
            raise FileNotFoundError(f"--reverse-dir が存在しません: {rev_dir}")
    else:
        rev_dir = find_reverse_analysis_dir(base)
        if rev_dir is None:
            raise FileNotFoundError("逆方向の analysis_csv フォルダが見つかりませんでした。--reverse-dir で明示指定してください。")
    print(f"[REV] analysis_csv directory: {rev_dir}")

    # 2) 順方向 CSV を決定
    if args.forward_file:
        fwd_csv = Path(args.forward_file).resolve()
        if not fwd_csv.exists():
            raise FileNotFoundError(f"--forward-file が見つかりません: {fwd_csv}")
    else:
        search_base = Path(args.forward_dir).resolve() if args.forward_dir else base
        fwd_csv = find_forward_csv(search_base)
        if fwd_csv is None:
            raise FileNotFoundError("順方向 CSV が見つかりませんでした。--forward-file で明示指定してください。")
    print(f"[FWD] summary file: {fwd_csv}")

    # 3) 逆方向サマリー再構築
    rev_summary = rebuild_reverse_summary(rev_dir)
    rev_out_path = out_root.parent / "reverse" / "Table_4.2.4.4R_summary_all_conditions_rebuilt.csv"
    rev_out_path.parent.mkdir(parents=True, exist_ok=True)
    rev_summary.to_csv(rev_out_path, index=False, encoding="utf-8-sig")
    print("[SAVE]", rev_out_path)

    # 4) 順方向横持ち化
    fwd_w = make_forward_wide(fwd_csv)

    # 5) マージ
    merged = (fwd_w.merge(rev_summary, on=["mode","index"], how="outer")
                     .sort_values(["mode","index"]).reset_index(drop=True))
    joined_csv = out_root / "Table_4.2.4.4_bidirectional_joined.csv"
    merged.to_csv(joined_csv, index=False, encoding="utf-8-sig")
    print("[SAVE]", joined_csv)

    # 6) 相関 & 図
    corr = compute_correlations(merged)
    corr_csv = out_root / "Table_4.2.4.4_bidirectional_correlations.csv"
    corr.to_csv(corr_csv, index=False, encoding="utf-8-sig")
    print("[SAVE]", corr_csv)
    save_core_scatter(merged, out_root)

    # 7) README と zip（任意）
    readme = {
        "base": str(base),
        "reverse_analysis_dir": str(rev_dir),
        "forward_csv": str(fwd_csv),
        "outputs": {
            "joined": str(joined_csv),
            "correlations": str(corr_csv),
            "figs": sorted([p.name for p in out_root.glob("Fig_4.2.4.4_bidirectional_*.png")]),
        },
        "note": "ARI_mean など forward 指標が欠損の多い場合、散布図は作られないことがあります。"
    }
    (out_root / "README_bidirectional_summary.json").write_text(
        json.dumps(readme, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print("[SAVE]", out_root / "README_bidirectional_summary.json")

    if args.save_zip:
        zip_path = out_root.with_suffix(".zip")
        import zipfile
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
            for p in out_root.rglob("*"):
                z.write(p, p.relative_to(out_root.parent))
        print("[ZIP ]", zip_path)

    print("\n✅ Done. outputs ->", out_root)

if __name__ == "__main__":
    main()

[REV] analysis_csv directory: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/reverse/analysis_csv
[FWD] summary file: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022/main_text/Table_4.2.4.4_OHFP-contrast_summary_k-le30.csv
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/reverse/Table_4.2.4.4R_summary_all_conditions_rebuilt.csv
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/bidirectional_summary/Table_4.2.4.4_bidirectional_joined.csv
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/bidirectional_summary/Table_4.2.4.4_bidirectional_correlations.csv
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/bidirectional_summary/README_bidirectional_summary.json

✅ Done. outputs -> /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/bidirectional_summar